# HW-12 · Perturbation: checking the map against the truth, not intuition

Homework for the module [«Explaining DL models via perturbation»](https://ai-interpretability.school).

All perturbation methods share one problem: **we do not know the true importance**. There is
nothing to compare the map with, and whether it is "good" is decided by eye.

Here that problem is gone. The data are synthetic: the class is determined by the brightness of
a 4×4 square in a known place, everything else is noise. The truth is known by construction, so
the map can be checked against it and turned into an honest number.

**What we do:** train a tiny convolutional network (three epochs on CPU, seconds), build three
maps — pixel-wise ablation, occlusion, RISE — and measure each by the ratio of mean brightness
inside the true square to the brightness outside. A random map gives the lower bound.

**The main task is not about the maps but about the metric.** At the end you will compute the
same maps with a second metric and see it declare a working method broken. Working out why is
the point of this homework.

## 1. Data, model, object

In [ ]:
import numpy as np, torch, torch.nn as nn

SEED = 0
torch.manual_seed(SEED); np.random.seed(SEED)
H = W = 16
PATCH = (slice(4, 8), slice(4, 8))   # правда: сигнал лежит здесь
N_TRAIN, N_TEST = 2000, 500

def make(n):
    x = np.random.randn(n, 1, H, W).astype(np.float32) * 0.5
    y = (np.random.rand(n) > 0.5).astype(np.int64)
    for i in range(n):
        if y[i] == 1:
            x[i, 0, PATCH[0], PATCH[1]] += 0.6
    return torch.from_numpy(x), torch.from_numpy(y)

xtr, ytr = make(N_TRAIN); xte, yte = make(N_TEST)

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1))
        self.head = nn.Linear(16, 2)
    def forward(self, x):
        return self.head(self.body(x).flatten(1))

net = Net(); opt = torch.optim.Adam(net.parameters(), lr=0.005)
lossf = nn.CrossEntropyLoss()
for epoch in range(3):
    perm = torch.randperm(N_TRAIN)
    for i in range(0, N_TRAIN, 64):
        idx = perm[i:i + 64]
        opt.zero_grad(); lossf(net(xtr[idx]), ytr[idx]).backward(); opt.step()
net.eval()
with torch.no_grad():
    acc = (net(xte).argmax(1) == yte).float().mean().item()
print(f'точность на тесте: {acc:.2f}')

In [ ]:
with torch.no_grad():
    probs = net(xte).softmax(1)

# Берём НЕ самый уверенный объект: у насыщенной модели softmax упирается в единицу,
# и любое вмешательство перестаёт что-либо менять — карта выходит равномерной кашей.
cands = [i for i in range(N_TEST) if yte[i] == 1 and probs[i, 1] > 0.5]
obj = min(cands, key=lambda i: abs(probs[i, 1].item() - 0.9))
x0, p0 = xte[obj:obj + 1], probs[obj, 1].item()
print(f'объект {obj}, уверенность {p0:.3f}')

def conf(batch):
    with torch.no_grad():
        return net(batch).softmax(1)[:, 1]

mask_true = np.zeros((H, W), dtype=bool); mask_true[PATCH] = True

## 2. Pixel-wise ablation

In [ ]:
abl = np.zeros((H, W), dtype=np.float32)
for i in range(H):
    for j in range(W):
        # ── Your code here ──
        # занулите пиксель (i, j) в копии x0 и запишите падение уверенности
        pass
print(f'макс вклад одного пикселя: {abl.max():.4f}')

## 3. Occlusion

In [ ]:
win, stride = 4, 2
pos = ...        # # ── Your code here ── сколько положений окна помещается по одной оси?
occ = np.zeros((H, W), dtype=np.float32); cnt = np.zeros((H, W), dtype=np.float32)
runs = 0
# ── Your code here ──
# пройдите окном по всем положениям, закрывая квадрат нулями,
# и накопите падение уверенности во все пиксели под окном
print(f'положений по оси: {pos}, прогонов: {runs}')

## 4. RISE

In [ ]:
def rise(n_masks, p=0.5, grid=4, seed=0):
    g = torch.Generator().manual_seed(seed)
    sal = torch.zeros(H, W)
    for _ in range(n_masks):
        # ── Your code here ──
        # сгенерируйте маску на сетке grid×grid, растяните до размера входа,
        # прогоните x0 под маской и накопите взвешенную маску
        pass
    return (sal / (n_masks * p)).numpy()   # не забудьте про нормировку

## 5. Checking against the truth

In [ ]:
def ratio(m):
    """Во сколько раз карта ярче внутри истинного патча, чем снаружи."""
    return m[mask_true].mean() / m[~mask_true].mean()

rng = np.random.default_rng(SEED)
rand = rng.random((H, W))
print(f'случайная карта: {ratio(rand):.2f}')
print(f'абляция:         {ratio(abl):.1f}')
print(f'окклюзия:        {ratio(occ):.1f}')
for n in (50, 200, 1000):
    print(f'RISE {n:>4} масок: {ratio(rise(n)):.2f}')

## 6. Same map, different metric

In [ ]:
# Ловушка, ради которой домашка и написана.
# Посчитайте ту же тройку карт другой метрикой — долей суммарной массы внутри патча:
def share(m):
    return m[mask_true].sum() / m.sum()

for name, m in (('случайная', rand), ('абляция', abl), ('окклюзия', occ),
                ('RISE 200', rise(200))):
    print(f'{name:>10}: доля массы {share(m):.2f}, отношение {ratio(m):.2f}')

**Task.** By the second metric RISE is indistinguishable from a random map, although by the first it was ahead. Does that mean the method does not work? Test the hypothesis with the next cell before answering.

In [ ]:
# Проверка: а различает ли модель маски, закрывающие патч?
g = torch.Generator().manual_seed(7)
hid, vis = [], []
for _ in range(300):
    small = (torch.rand(1, 1, 4, 4, generator=g) < 0.5).float()
    m = torch.nn.functional.interpolate(small, size=(H, W), mode='bilinear',
                                        align_corners=False)
    c = conf(x0 * m).item()
    frac = m[0, 0][torch.from_numpy(mask_true)].mean().item()
    (vis if frac > 0.7 else hid if frac < 0.3 else []).append(c)
print(f'патч виден  (n={len(vis)}): средняя уверенность {np.mean(vis):.3f}')
print(f'патч закрыт (n={len(hid)}): средняя уверенность {np.mean(hid):.3f}')

**The takeaway.** The metric is as much a choice by the author of an explanation as the baseline or the window size. A badly chosen metric buries a working method, and the only way to notice is the check you have just run.